In [0]:
%run "./01_logging_utils.py"

In [0]:
import time
from datetime import datetime

Generating one ID for the overall pipeline execution

In [0]:
pipeline_id = new_batch_id("pipeline")
pipeline_start = datetime.now()
pipeline_start_clock = time.time()

Reusing the incremental file path defined in 00_config_schemas

In [0]:
batch_id = new_batch_id("inc")

Running notebooks in dependency order

In [0]:

steps = [
    ("03_bronze_incremental.py", {"path": INC_FILE, "batch_id": batch_id}),
    ("04_silver_transform.py", {}),
    ("08_gold_setup", {}),
    ("09_gold_dimensions", {}),
    ("10_gold_facts", {}),
    ("11_gold_optimize", {}),
    ("13_validation", {}),
    ("14_bi_views", {})
]

failed_step = None

print(f"Pipeline ID: {pipeline_id}")
print(f"Incremental file: {INC_FILE}")
print(f"Batch ID: {batch_id}")
print(f"Total steps: {len(steps)}")

try:
    for notebook_name, arguments in steps:
        step_start = time.time()
        print(f"\nStarting: {notebook_name}")

        result = dbutils.notebook.run(
            f"./{notebook_name}",
            timeout_seconds=0,
            arguments=arguments
        )

        elapsed = round(time.time() - step_start, 2)
        print(f"Completed: {notebook_name} | Duration: {elapsed}s")

        if result:
            print(f"Output: {result}")

    duration = round(time.time() - pipeline_start_clock, 2)

    log_run(
        pipeline_id,
        "pipeline",
        f"Full pipeline | batch_id={batch_id} | steps={len(steps)} | duration_seconds={duration}",
        pipeline_start,
        datetime.now(),
        "SUCCESS"
    )

    print("\nPIPELINE COMPLETED SUCCESSFULLY")
    print(f"Pipeline ID: {pipeline_id}")
    print(f"Duration: {duration}s")

except Exception as e:
    failed_step = notebook_name
    duration = round(time.time() - pipeline_start_clock, 2)

    log_run(
        pipeline_id,
        "pipeline",
        f"Full pipeline | batch_id={batch_id} | failed_step={failed_step} | duration_seconds={duration}",
        pipeline_start,
        datetime.now(),
        "FAILED",
        err=str(e)[:500]
    )

    print(f"\nPIPELINE FAILED at: {failed_step}")
    print(f"Error: {str(e)[:500]}")
    print(f"Duration before failure: {duration}s")

    raise



display(
    spark.sql(f"""
        SELECT
            run_id,
            layer,
            file_or_params,
            start_time,
            end_time,
            status,
            rows_inserted,
            rows_updated,
            error_message
        FROM {T_LOGS}
        WHERE run_id = '{pipeline_id}'
    """)
)
